# Create and run QeMCMC circuit from a graph


In [1]:
from qemcmc import mis_energy_model, random_bitstring, build_proposer_qc
from qemcmc import active_qubit_count
from qemcmc import load_dimacs_edge

from qiskit_ibm_runtime import SamplerV2
from qiskit_ibm_runtime.fake_provider import FakeFez
from qiskit.transpiler import generate_preset_pass_manager
from qiskit_aer import AerSimulator
from qiskit.visualization import plot_histogram

Load a graph from the assets directory.

In [2]:
# import rustworkx as rx
# graph = rx.generators.path_graph(6)
graph = load_dimacs_edge("ibm32.gph")

model, _ = mis_energy_model(graph, lam=2.0)
initial_spin_state = random_bitstring(model.n_spins, rng=123)
circuit = build_proposer_qc(model, initial_spin_state, gamma=0.5, time=6, delta_time=0.8)

If we have no measurements, there is no output to collect from shots.

In [3]:
circuit.measure_all()


print("Number of qubits in circuit ", circuit.num_qubits)

Number of qubits in circuit  32


Number of qubits in device must be greater or equal to what we have.

In [4]:
backend = FakeFez()
pm = generate_preset_pass_manager(backend=backend, optimization_level=1)

print("Transpiling...")
isa_circuit = pm.run(circuit)

Transpiling...


The width of the transpiled circuit is equal to the width of the device.

In [5]:
print("Number of qubits in isa_qc ", isa_circuit.num_qubits)

print("Number of active qubits in isa_qc ", active_qubit_count(isa_circuit))

sampler = SamplerV2(backend)

print("Running circuit...")
job = sampler.run([isa_circuit])

print("Fetching result...")
pub_result = job.result()[0]

print("Getting counts...")
counts = pub_result.data.meas.get_counts()

print("input spin state:", initial_spin_state)
# print("counts:", counts)

Number of qubits in isa_qc  156
Number of active qubits in isa_qc  37
Running circuit...
Fetching result...


Simulation failed and returned the following error message:
ERROR:  [Experiment 0] a circuit requires more memory than max_memory_mb.


ValueError: could not broadcast input array from shape (0,4) into shape (1024,4)

Plot a histogram of the freqencies of counts collected

In [ ]:
plot_histogram(counts)

# Show plot on the command line
# import matplotlib.pyplot as plt
# plt.show()